# Hiver Support AI — Exploratory Data Analysis
## Phase 2: Data Engineering & Analytics

This notebook explores the processed conversation dataset produced by the Phase 2 preprocessing pipeline.
All analysis logic is imported from `src/analysis/` modules — this notebook only loads data, calls functions, and renders results.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config
from src.analysis.dataset_overview import compute_overview
from src.analysis.conversation_analysis import analyze_conversations
from src.analysis.intent_discovery import analyze_customer_language, discover_intents, analyze_escalation_patterns
from src.analysis.visualization import generate_all_charts

print('Imports complete.')

## 1. Load Configuration & Data

In [ ]:
config = load_config()

df = pd.read_csv(
    config.processed_conversations_path,
    dtype={'tweet_id': str, 'author_id': str, 'conversation_id': str},
    parse_dates=['created_at'],
    low_memory=False,
)

print(f'Loaded {len(df):,} messages across {df["conversation_id"].nunique():,} conversations')
print(f'Brand: {config.selected_brand}')
df.head()

## 2. Dataset Overview

In [ ]:
overview_stats = compute_overview(df, config)

# Display key metrics
for section, data in overview_stats.items():
    if section.startswith('_'):
        continue
    print(f'\n--- {section} ---')
    if isinstance(data, dict):
        for k, v in data.items():
            print(f'  {k}: {v}')

## 3. Conversation Structure Analysis

In [ ]:
conversation_stats = analyze_conversations(df, config)

print(f'Single-turn: {conversation_stats["turn_structure"]["single_turn_count"]:,}')
print(f'Multi-turn:  {conversation_stats["turn_structure"]["multi_turn_count"]:,}')
print(f'\nLength buckets:')
for bucket, count in conversation_stats.get('length_buckets', {}).items():
    print(f'  {bucket}: {count:,}')

## 4. Customer Language Analysis

In [ ]:
language_stats = analyze_customer_language(df, config)

print('Top 15 customer keywords:')
for kw in language_stats.get('top_keywords', [])[:15]:
    print(f'  {kw["word"]:<20} {kw["count"]:>6,}')

qc = language_stats.get('question_vs_complaint', {})
print(f'\nQuestions: {qc.get("questions", 0):,} ({qc.get("questions_pct", 0)}%)')
print(f'Complaints: {qc.get("complaints", 0):,} ({qc.get("complaints_pct", 0)}%)')

## 5. Intent Discovery

In [ ]:
intent_stats = discover_intents(df, config)

for cluster in intent_stats.get('clusters', []):
    print(f'\nTheme: {cluster["theme_label"]}')
    print(f'  Size: {cluster["size"]:,} ({cluster["frequency_pct"]}%)')
    top_words = ', '.join(kw['word'] for kw in cluster.get('top_keywords', [])[:5])
    print(f'  Keywords: {top_words}')
    if cluster.get('representative_messages'):
        print(f'  Sample: "{cluster["representative_messages"][0]["text"][:80]}..."')

## 6. Escalation Patterns

In [ ]:
escalation_stats = analyze_escalation_patterns(df, config)

for key in ['repeated_followups', 'unresolved']:
    if key in escalation_stats:
        data = escalation_stats[key]
        print(f'{key}: {data["count"]:,} ({data["pct"]}%)')

overall = escalation_stats.get('overall_estimate', {})
print(f'\nOverall with any signal: {overall.get("conversations_with_any_signal", 0):,} ({overall.get("pct", 0)}%)')

## 7. Generate & Save All Charts

In [ ]:
chart_paths = generate_all_charts(
    overview_stats=overview_stats,
    conversation_stats=conversation_stats,
    language_stats=language_stats,
    intent_stats=intent_stats,
    escalation_stats=escalation_stats,
    config=config,
)

print(f'Saved {len(chart_paths)} charts to {config.eda_output_dir}')
for p in chart_paths:
    print(f'  {p.name}')

## Summary

All analysis functions are imported from `src/analysis/`. No business logic is duplicated in this notebook.

**Next Steps**: Use `annotation_ready.csv` for Phase 3 labeling tasks.